In [1]:
import logging

logging.getLogger('fastf1').setLevel(logging.ERROR)

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
import fastf1
import pandas as pd
import numpy as np
from pathlib import Path
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

fastf1.Cache.enable_cache('cache')

In [3]:
df_drivers = pd.read_csv('data/Model2/drivers.csv').convert_dtypes()
df_laps = pd.read_csv('data/Model2/laps.csv').convert_dtypes()
df_track_status = pd.read_csv('data/Model2/track_status.csv').convert_dtypes()
df_weather = pd.read_csv('data/Model2/weather.csv').convert_dtypes()

C:\Users\ognjeni\AppData\Local\Temp\ipykernel_7396\2993207287.py:2: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df_laps = pd.read_csv('data/Model2/laps.csv').convert_dtypes()


In [4]:
for col in df_laps.select_dtypes(include=['string','boolean']).columns:
    if df_laps[col].dtype.name == 'boolean':
        df_laps[col] = df_laps[col].astype('float')
    else:
        df_laps[col] = df_laps[col].astype('object')

In [5]:
df_laps = df_laps.replace({pd.NA: np.nan})

In [6]:
df_laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Season,Round,EventName
0,0 days 00:34:06.302000,GAS,10,0 days 00:01:19.106000,1,1,NaN,NaN,NaN,0 days 00:00:33.352000,0 days 00:00:23.247000,NaN,0 days 00:33:43.079000,0 days 00:34:06.511000,309,217,274,290,0.0,MEDIUM,1,1.0,AlphaTauri,0 days 00:32:47.006000,<NA>,1,12,<NA>,<NA>,0.0,0.0,2020,0,Pre-Season Test 1
1,0 days 00:35:18.714000,GAS,10,0 days 00:01:12.412000,2,1,NaN,NaN,0 days 00:00:17.960000,0 days 00:00:32.228000,0 days 00:00:22.224000,0 days 00:34:24.262000,0 days 00:34:56.490000,0 days 00:35:18.714000,282,226,270,292,1.0,MEDIUM,2,1.0,AlphaTauri,0 days 00:34:06.302000,<NA>,1,12,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1
2,0 days 00:36:30.025000,GAS,10,0 days 00:01:11.311000,3,1,NaN,NaN,0 days 00:00:17.513000,0 days 00:00:31.717000,0 days 00:00:22.081000,0 days 00:35:36.227000,0 days 00:36:07.944000,0 days 00:36:30.025000,307,224,277,311,1.0,MEDIUM,3,1.0,AlphaTauri,0 days 00:35:18.714000,<NA>,1,12,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1
3,0 days 00:37:40.750000,GAS,10,0 days 00:01:10.725000,4,1,NaN,NaN,0 days 00:00:17.410000,0 days 00:00:31.510000,0 days 00:00:21.805000,0 days 00:36:47.435000,0 days 00:37:18.945000,0 days 00:37:40.750000,309,218,275,309,1.0,MEDIUM,4,1.0,AlphaTauri,0 days 00:36:30.025000,<NA>,1,12,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1
4,0 days 00:38:52.261000,GAS,10,0 days 00:01:11.511000,5,1,NaN,NaN,0 days 00:00:17.464000,0 days 00:00:31.827000,0 days 00:00:22.220000,0 days 00:37:58.214000,0 days 00:38:30.041000,0 days 00:38:52.261000,303,219,268,296,0.0,MEDIUM,5,1.0,AlphaTauri,0 days 00:37:40.750000,<NA>,1,12,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1


In [11]:
df_drivers.tail()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps,Season,Round
2114,77,V BOTTAS,BOT,bottas,Kick Sauber,52e252,sauber,Valtteri,Bottas,Valtteri Bottas,https://media.formula1.com/d_driver_fallback_i...,FIN,16,16,19,<NA>,<NA>,<NA>,0 days 00:00:57.829000,Lapped,0.0,61,2024,18
2115,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,FRA,17,17,18,<NA>,<NA>,<NA>,0 days 00:00:59.059000,Lapped,0.0,61,2024,18
2116,3,D RICCIARDO,RIC,ricciardo,RB,6692FF,rb,Daniel,Ricciardo,Daniel Ricciardo,https://media.formula1.com/d_driver_fallback_i...,AUS,18,18,16,<NA>,<NA>,<NA>,0 days 00:01:29.796000,Lapped,0.0,61,2024,18
2117,20,K MAGNUSSEN,MAG,kevin_magnussen,Haas F1 Team,B6BABD,haas,Kevin,Magnussen,Kevin Magnussen,https://media.formula1.com/d_driver_fallback_i...,DEN,19,19,14,<NA>,<NA>,<NA>,<NA>,Retired,0.0,57,2024,18
2118,23,A ALBON,ALB,albon,Williams,64C4FF,williams,Alexander,Albon,Alexander Albon,https://media.formula1.com/d_driver_fallback_i...,THA,20,R,11,<NA>,<NA>,<NA>,<NA>,Retired,0.0,15,2024,18


In [9]:
df_track_status.head()

,Time,Status,Message,Season,Round
0,0 days 00:07:56.420000,1,AllClear,2020,0
1,0 days 00:56:21.333000,2,Yellow,2020,0
2,0 days 00:56:30.327000,1,AllClear,2020,0
3,0 days 01:01:52.836000,2,Yellow,2020,0
4,0 days 01:02:31.497000,4,SCDeployed,2020,0


In [10]:
df_weather.head()

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,Season,Round
0,0 days 00:00:38.669000,27.5,35.1,939.4,False,55.6,99,1.7,2020,0
1,0 days 00:01:38.684000,27.1,36.5,939.4,False,55.1,106,1.9,2020,0
2,0 days 00:02:38.655000,27.0,36.2,939.3,False,54.2,111,2.1,2020,0
3,0 days 00:03:38.669000,26.9,36.1,939.3,False,54.2,165,1.0,2020,0
4,0 days 00:04:38.678000,27.2,35.8,939.3,False,54.2,83,1.2,2020,0


In [12]:
df_laps.dtypes

Time                   object
Driver                 object
DriverNumber            Int64
LapTime                object
LapNumber               Int64
Stint                   Int64
PitOutTime             object
PitInTime              object
Sector1Time            object
Sector2Time            object
Sector3Time            object
Sector1SessionTime     object
Sector2SessionTime     object
Sector3SessionTime     object
SpeedI1                 Int64
SpeedI2                 Int64
SpeedFL                 Int64
SpeedST                 Int64
IsPersonalBest        float64
Compound               object
TyreLife                Int64
FreshTyre             float64
Team                   object
LapStartTime           object
LapStartDate            Int64
TrackStatus             Int64
Position                Int64
Deleted                 Int64
DeletedReason           Int64
FastF1Generated       float64
IsAccurate            float64
Season                  Int64
Round                   Int64
EventName 

# Target creation

In [7]:
df_laps['LapTime'] = pd.to_timedelta(df_laps['LapTime'])
df_laps['LapTime_ms'] = df_laps['LapTime'].dt.total_seconds() * 1000

In [9]:
df_laps = df_laps.sort_values(['Season', 'Round', 'EventName', 'Driver', 'LapNumber'])

In [10]:
df_laps['target'] = df_laps['LapTime_ms'].shift(-1)

In [11]:
df_laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Season,Round,EventName,LapTime_ms,target
236,0 days 00:34:02.300000,ALB,23,0 days 00:01:15.104000,1,1,NaN,NaN,NaN,0 days 00:00:32.211000,0 days 00:00:21.871000,NaN,0 days 00:33:40.459000,0 days 00:34:02.513000,291,232,275,294,0.0,SOFT,4,0.0,Red Bull Racing,0 days 00:32:47.006000,<NA>,1,4,<NA>,<NA>,0.0,0.0,2020,0,Pre-Season Test 1,75104.0,70362.0
237,0 days 00:35:12.662000,ALB,23,0 days 00:01:10.362000,2,1,NaN,NaN,0 days 00:00:17.405000,0 days 00:00:31.428000,0 days 00:00:21.529000,0 days 00:34:19.721000,0 days 00:34:51.149000,0 days 00:35:12.678000,297,234,276,291,1.0,SOFT,5,0.0,Red Bull Racing,0 days 00:34:02.300000,<NA>,1,4,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1,70362.0,69579.0
238,0 days 00:36:22.241000,ALB,23,0 days 00:01:09.579000,3,1,NaN,NaN,0 days 00:00:17.231000,0 days 00:00:30.945000,0 days 00:00:21.403000,0 days 00:35:29.909000,0 days 00:36:00.854000,0 days 00:36:22.257000,321,233,271,315,1.0,SOFT,6,0.0,Red Bull Racing,0 days 00:35:12.662000,<NA>,1,3,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1,69579.0,69853.0
239,0 days 00:37:32.094000,ALB,23,0 days 00:01:09.853000,4,1,NaN,NaN,0 days 00:00:17.637000,0 days 00:00:30.971000,0 days 00:00:21.245000,0 days 00:36:39.894000,0 days 00:37:10.865000,0 days 00:37:32.110000,287,231,273,285,0.0,SOFT,7,0.0,Red Bull Racing,0 days 00:36:22.241000,<NA>,1,3,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1,69853.0,69412.0
240,0 days 00:38:41.506000,ALB,23,0 days 00:01:09.412000,5,1,NaN,NaN,0 days 00:00:17.480000,0 days 00:00:30.845000,0 days 00:00:21.087000,0 days 00:37:49.590000,0 days 00:38:20.435000,0 days 00:38:41.522000,294,234,274,288,1.0,SOFT,8,0.0,Red Bull Racing,0 days 00:37:32.094000,<NA>,1,3,<NA>,<NA>,0.0,1.0,2020,0,Pre-Season Test 1,69412.0,69525.0


In [12]:
df_laps.describe()

,DriverNumber,LapTime,LapNumber,Stint,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,TyreLife,FreshTyre,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Season,Round,LapTime_ms,target
count,115530.0,113394,115530.0,115530.0,98457.0,115276.0,111182.0,106425.0,115372.000000,114906.0,115530.000000,0.0,115507.0,115324.0,0.0,0.0,115530.000000,115530.000000,115530.0,115530.0,1.133940e+05,1.133930e+05
mean,27.600744,0 days 00:01:31.230252579,30.562443,2.118021,254.077343,243.407309,260.718273,291.972046,0.211758,14.533175,0.760287,<NA>,10.964323,9.695796,<NA>,<NA>,0.001783,0.857111,2021.987856,9.946196,9.123025e+04,9.123039e+04
std,24.49966,0 days 00:00:32.170024135,18.467209,0.995796,50.148389,44.84517,38.780221,34.736029,0.408556,10.226007,0.426910,<NA>,115.651742,5.389482,<NA>,<NA>,0.042189,0.349961,1.348173,6.351694,3.217002e+04,3.217013e+04
min,1.0,0 days 00:00:55.404000,1.0,1.0,38.0,35.0,1.0,36.0,0.000000,1.0,0.000000,<NA>,1.0,1.0,<NA>,<NA>,0.000000,0.000000,2020.0,0.0,5.540400e+04,5.540400e+04
25%,10.0,0 days 00:01:20.712000,15.0,1.0,217.0,219.0,245.0,287.0,0.000000,7.0,1.000000,<NA>,1.0,5.0,<NA>,<NA>,0.000000,1.000000,2021.0,5.0,8.071200e+04,8.071200e+04
50%,20.0,0 days 00:01:29.697000,30.0,2.0,272.0,253.0,272.0,298.0,0.000000,13.0,1.000000,<NA>,1.0,10.0,<NA>,<NA>,0.000000,1.000000,2022.0,10.0,8.969700e+04,8.969700e+04
75%,44.0,0 days 00:01:39.454750,45.0,3.0,289.0,275.0,286.0,309.0,0.000000,20.0,1.000000,<NA>,1.0,14.0,<NA>,<NA>,0.000000,1.000000,2023.0,15.0,9.945475e+04,9.945500e+04
max,99.0,0 days 00:42:06.253000,87.0,8.0,359.0,350.0,355.0,362.0,1.000000,78.0,1.000000,<NA>,16724.0,20.0,<NA>,<NA>,1.000000,1.000000,2024.0,22.0,2.526253e+06,2.526253e+06


We are dropping na values for target, because, this data represent last laps. Last laps does not have next time lap.

In [19]:
df_laps = df_laps.dropna(subset=['target'])

In [20]:
df = df_laps

# First model

In [29]:
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [15]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

### Some helper functions

In [16]:
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.cols, errors='ignore')

In [17]:
def na_to_binary(X):
    return X.notna().astype(int)

In [21]:
class TimeConverter(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col] = pd.to_timedelta(X[col], errors='coerce').dt.total_seconds()
        return X

#### Eval function

In [23]:
# Function for evaluation shold be created.
def eval(df, pipelines, target_col, feature_cols):
    seasons = sorted(df['Season'].unique())
    results = []

    for i in range(1, len(seasons)):
        train_seasons = seasons[:i]
        test_season = seasons[i]

        train_df = df[df['Season'].isin(train_seasons)]
        test_df = df[df['Season'] == test_season]

        X_train = train_df[feature_cols]
        y_train = train_df[target_col]

        X_test = test_df[feature_cols]
        y_test = test_df[target_col]

        print(f"\n=========== ITERATION {i} | TEST SEASON {test_season} ===========")

        models = clone(pipelines)
        for name, model in models.items():
            print(f"\nTraining {name}...")
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            mae = mean_absolute_error(y_test, y_pred)
            rmse = root_mean_squared_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            results.append({
                'season': test_season,
                'model': name,
                'MAE': mae,
                'RMSE': rmse, 
                'R2': r2
            })

            print(f"{name} -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.3f}")

    return results

### Data

In [27]:
numerical_cols = ['LapTime', 'LapNumber', 'Stint', 'TyreLife', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1',
                 'SpeedI2', 'SpeedFL', 'SpeedST', 'TrackStatus']
categorical_cols = ['Driver', 'Compound', 'Team', 'FreshTyre', 'EventName']

In [24]:
cols_to_drop = ['Time', 'DriverNumber', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'IsPersonalBest', 
                'LapStartTime', 'LapStartDate', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate',
                'Season', 'Round', 'NextPosition']

In [25]:
binary_features = ['PitInTime', 'PitOutTime']

In [28]:
preprocessor = Pipeline(steps=[
    ('drop_cols', DropColumns(cols_to_drop)),
    ('convertor', TimeConverter([
        'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time'
    ])),
    ('preprocess', ColumnTransformer(
        transformers=[
            
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='mean')),
                ('scaler', StandardScaler())
            ]), numerical_cols),
            
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))
            ]), categorical_cols),
            
            ('bin_transform', Pipeline([
                ('to_binary', FunctionTransformer(na_to_binary))
            ]), binary_features)
        ]
    ))
])

In [31]:
pipeline_lr_I = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])

pipeline_rf_I = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

pipeline_xgb_I = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])

In [32]:
pipelines = {
    "Linear Regression": pipeline_lr_I,
    "Random Forest": pipeline_rf_I,
    "XGBoost": pipeline_xgb_I
}

In [33]:
target_col = 'target'
feature_cols = numerical_cols + categorical_cols + binary_features

In [34]:
results = eval(df, pipelines, target_col, feature_cols)


=========== ITERATION 1 | TEST SEASON 2021 ===========

Training Linear Regression...
Linear Regression -> MAE: 6447.04, RMSE: 10108.07, R2: 0.517

Training Random Forest...
Random Forest -> MAE: 5989.36, RMSE: 9766.94, R2: 0.549

Training XGBoost...
XGBoost -> MAE: 4719.81, RMSE: 8017.17, R2: 0.696

=========== ITERATION 2 | TEST SEASON 2022 ===========

Training Linear Regression...
Linear Regression -> MAE: 4104.32, RMSE: 8392.27, R2: 0.713

Training Random Forest...
Random Forest -> MAE: 4244.73, RMSE: 8611.76, R2: 0.698

Training XGBoost...
XGBoost -> MAE: 3763.53, RMSE: 7878.04, R2: 0.747

=========== ITERATION 3 | TEST SEASON 2023 ===========

Training Linear Regression...
Linear Regression -> MAE: 3837.31, RMSE: 7094.82, R2: 0.730

Training Random Forest...
Random Forest -> MAE: 2849.35, RMSE: 6260.31, R2: 0.790

Training XGBoost...
XGBoost -> MAE: 2423.39, RMSE: 5351.77, R2: 0.846

=========== ITERATION 4 | TEST SEASON 2024 ===========

Training Linear Regression...
Linear Re